# Column hardware: diameter, tray hydraulics, packing and cost

`difflow.units.column_design` sizes trayed and packed columns and feeds `column_cost`.
**The correlation constants were written from memory and are unverified** (chart-reading accuracy,
10-20 %; the entrainment function and the GPDC pressure-drop shape are placeholders): read the
docstrings and the *Column Hardware* section of the unit-operations docs before using a number.

In [1]:
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
from difflow.units.column_design import *
from difflow.mass_transfer import PACKINGS

## 1. Tray diameter at 80 % of Fair flood, and its scaling

In [2]:
col = dict(V_molar=100.0, MW_V=80.0, rho_V=2.9, L_molar=80.0, MW_L=85.0, rho_L=790.0, sigma=0.021)
D, info = tray_column_diameter(**col, tray_spacing=0.6, fraction_flood=0.8)
print({k: round(float(v), 3) for k, v in info.items()}, float(D))
for ff in (0.6, 0.7, 0.8, 0.9):
    print(ff, round(float(tray_column_diameter(**col, fraction_flood=ff)[0]), 3))

{'F_LV': 0.051, 'C_SB': 0.107, 'u_flood': 1.761, 'u_operating': 1.409, 'Q_V': 2.759, 'A_column': 2.176, 'A_net': 1.958, 'A_downcomer': 0.218, 'A_active': 1.74, 'percent_flood': 80.0} 1.6643561054494225
0.6 1.922
0.7 1.779
0.8 1.664
0.9 1.569


## 2. Tray pressure drop and the weeping margin at turndown

In [3]:
Q_V, Q_L = float(info["Q_V"]), 80 * 85e-3 / 790
dP, h = tray_pressure_drop(Q_V, Q_L, 2.9, 790.0, D)
print(float(dP), {k: round(float(h[k]), 1) for k in ("dP_dry", "dP_liquid", "dP_residual")})
for turn in (1.0, 0.7, 0.5, 0.3):
    print(turn, round(float(weep_point(turn * Q_V, Q_L, 2.9, D)[0]), 2))

1274.8130339798904 {'dP_dry': 569.2, 'dP_liquid': 583.1, 'dP_residual': 122.6}
1.0 1.32
0.7 0.62
0.5 0.16
0.3 -0.3


## 3. Whole column and the gradient of cost with respect to the fraction of flood

In [4]:
def cost(ff):
    D, _ = tray_column_diameter(**col, fraction_flood=ff)
    s = size_column(15.0, oconnell_efficiency(2.4, 0.3), D, tray_spacing=0.6)
    return column_hardware_cost(s)["total"]
print(float(cost(0.8)), float(jax.grad(cost)(0.8)))

159403.97490847894 -149630.57300415242


## 4. Packed column at 70 % of flood

In [5]:
pk = PACKINGS["pall_metal_25"]
air = dict(V_molar=100.0, MW_V=29.0, rho_V=1.2, L_molar=500.0, MW_L=18.0, rho_L=1000.0, mu_L=1e-3)
Dp, ip = packed_column_diameter(**air, packing=pk)
print(float(Dp), {k: round(float(v), 4) for k, v in ip.items()})

1.3582586415975009 {'F_LV': 0.1076, 'G_flood': 2.8592, 'G': 2.0014, 'u_flood': 2.3827, 'u_operating': 1.6679, 'dP_per_m': 619.1338, 'dP_flood_per_m': 1568.7852, 'wetting_rate': 0.0, 'wetting_margin': 0.2632, 'percent_flood': 70.0}
